# Electronic Waste Detection and Classification

## Phase 3 — YOLO26n Model Setup

### Project Objective

The objective of this project is to develop an object detection system capable of detecting and classifying electronic waste objects from images and camera input.

The system will identify electronic waste objects using:

- Bounding boxes
- Class labels
- Confidence scores

### Selected Model

YOLO26n is selected as the object detection model for this project.

YOLO26n is the lightweight variant of the YOLO26 family and is selected because the project requires an efficient object detection model that can later support image, video, and real-time camera inference while keeping computational requirements relatively moderate.

The model will be fine-tuned using the prepared electronic waste dataset containing 37 object classes.

### Dataset

The dataset was obtained from the Roboflow Balanced E-Waste Dataset.

The original dataset was downloaded in TensorFlow format. During the dataset preparation phase, the original annotations were converted into YOLO-compatible annotation format and organized into training, validation, and testing subsets.

### Project Workflow

1. Dataset Analysis
2. Dataset Preparation
3. YOLO26n Model Setup
4. YOLO26n Model Training
5. Model Evaluation
6. Image and Video Testing
7. Real-Time Camera Demonstration
8. Final Results and Documentation

## 3.1 Environment and Library Setup

The Ultralytics package provides the Python interface required to load, train, validate, and perform inference using YOLO26 models.

A dedicated Python virtual environment is used to maintain project dependencies and improve reproducibility.

The environment is verified before model training begins.

In [15]:
%pip install -U ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import sys
import torch
import ultralytics

print("=" * 70)
print("ENVIRONMENT INFORMATION")
print("=" * 70)

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nPyTorch version:")
print(torch.__version__)

print("\nUltralytics version:")
print(ultralytics.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU available to PyTorch: No")
    print("Training environment: CPU")

print("=" * 70)

ENVIRONMENT INFORMATION
Python version:
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]

Python executable:
c:\Users\USER\Desktop\computer vision\.venv\Scripts\python.exe

PyTorch version:
2.14.0+cpu

Ultralytics version:
8.4.153

CUDA available: False
GPU available to PyTorch: No
Training environment: CPU


## 3.2 Hardware Verification

The available hardware is recorded before model development because hardware resources affect training time, batch size, memory requirements, and inference performance.

The project computer has an Intel integrated graphics processor rather than an NVIDIA CUDA GPU.

Therefore, the lightweight YOLO26n model is selected to reduce computational requirements compared with larger YOLO26 variants.

The model can later be trained using a GPU-enabled environment if required.

In [17]:
import platform
import psutil
import torch

print("=" * 70)
print("HARDWARE INFORMATION")
print("=" * 70)

print("Operating System:")
print(platform.platform())

print("\nProcessor:")
print(platform.processor())

print("\nCPU:")
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))

memory_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"RAM: {memory_gb:.2f} GB")

print("\nPyTorch:")
print("Version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("CUDA GPU available to PyTorch: No")

print("=" * 70)

HARDWARE INFORMATION
Operating System:
Windows-11-10.0.26200-SP0

Processor:
Intel64 Family 6 Model 140 Stepping 1, GenuineIntel

CPU:
Physical cores: 4
Logical cores: 8
RAM: 15.70 GB

PyTorch:
Version: 2.14.0+cpu
CUDA available: False
CUDA GPU available to PyTorch: No


## 3.3 Locate the Prepared YOLO Dataset

The dataset preparation phase produced a YOLO-compatible dataset.

The prepared dataset follows the structure:

```text
YOLO Dataset
│
├── images
│   ├── train
│   ├── val
│   └── test
│
├── labels
│   ├── train
│   ├── val
│   └── test
│
└── data.yaml

In [18]:



# Cell 8 — Code


from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

# Search for data.yaml
yaml_files = list(PROJECT_ROOT.rglob("data.yaml"))

print("\nFound data.yaml files:")

for yaml_file in yaml_files:
    print(" -", yaml_file)

if not yaml_files:
    raise FileNotFoundError(
        "No data.yaml file was found inside the project."
    )

DATA_YAML = yaml_files[0]

print("\nSelected dataset configuration:")
print(DATA_YAML)

Current working directory:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\notebooks

Project root:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-

Found data.yaml files:
 - c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml

Selected dataset configuration:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\data.yaml


In [19]:
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

print("=" * 70)
print("YOLO DATASET CONFIGURATION")
print("=" * 70)

for key, value in data_config.items():
    print(f"{key}: {value}")

print("=" * 70)

YOLO DATASET CONFIGURATION
path: .
train: images/train
val: images/val
test: images/test
nc: 37
names: ['Battery', 'Blood-Pressure-Monitor', 'Boiler', 'Clothes-Iron', 'Coffee-Machine', 'Computer-Keyboard', 'Computer-Mouse', 'Cooling-Display', 'Desktop-PC', 'Digital-Oscilloscope', 'Drone', 'Electric-Guitar', 'Electronic-Keyboard', 'Flashlight', 'Flat-Panel-Monitor', 'Flat-Panel-TV', 'Glucose-Meter', 'HDD', 'Laptop', 'Microwave', 'Music-Player', 'Oven', 'PCB', 'Photovoltaic-Panel', 'Projector', 'Refrigerator', 'Rotary-Mower', 'Router', 'Server', 'Smartphone', 'Smoke-Detector', 'Straight-Tube-Fluorescent-Lamp', 'Street-Lamp', 'TV-Remote-Control', 'Telephone-Set', 'USB-Flash-Drive', 'Washing-Machine']


## 3.4 Verify Dataset Class Configuration

The dataset contains 37 electronic waste object classes.

The class configuration is checked programmatically to ensure that:

- The number of classes is correct.
- All class names are available.
- The class ordering is consistent with the dataset preparation phase.

A fixed class mapping is important because YOLO label files use numerical class IDs.

In [20]:
names = data_config.get("names")
nc = data_config.get("nc")

print("=" * 70)
print("CLASS CONFIGURATION")
print("=" * 70)

print("Number of classes (nc):", nc)
print("Number of class names:", len(names))

print("\nClass mapping:")

for class_id, class_name in enumerate(names):
    print(f"{class_id:2d}: {class_name}")

print("=" * 70)

assert nc == 37, f"Expected 37 classes, but found {nc}"
assert len(names) == 37, f"Expected 37 class names, but found {len(names)}"

print("\n✓ Class configuration is correct.")

CLASS CONFIGURATION
Number of classes (nc): 37
Number of class names: 37

Class mapping:
 0: Battery
 1: Blood-Pressure-Monitor
 2: Boiler
 3: Clothes-Iron
 4: Coffee-Machine
 5: Computer-Keyboard
 6: Computer-Mouse
 7: Cooling-Display
 8: Desktop-PC
 9: Digital-Oscilloscope
10: Drone
11: Electric-Guitar
12: Electronic-Keyboard
13: Flashlight
14: Flat-Panel-Monitor
15: Flat-Panel-TV
16: Glucose-Meter
17: HDD
18: Laptop
19: Microwave
20: Music-Player
21: Oven
22: PCB
23: Photovoltaic-Panel
24: Projector
25: Refrigerator
26: Rotary-Mower
27: Router
28: Server
29: Smartphone
30: Smoke-Detector
31: Straight-Tube-Fluorescent-Lamp
32: Street-Lamp
33: TV-Remote-Control
34: Telephone-Set
35: USB-Flash-Drive
36: Washing-Machine

✓ Class configuration is correct.


## 3.5 Verify YOLO Dataset Structure

The prepared dataset directories are checked before loading the detection model.

The following directories are required:

```text
images/train
images/val
images/test

labels/train
labels/val
labels/test

In [21]:



# Cell 13 — Code


dataset_root = DATA_YAML.parent

train_path = Path(data_config["train"])
val_path = Path(data_config["val"])
test_path = Path(data_config["test"])

# Resolve relative paths
if not train_path.is_absolute():
    train_path = dataset_root / train_path

if not val_path.is_absolute():
    val_path = dataset_root / val_path

if not test_path.is_absolute():
    test_path = dataset_root / test_path

print("=" * 70)
print("YOLO DATASET STRUCTURE")
print("=" * 70)

print("Dataset root:")
print(dataset_root)

print("\nTrain images:")
print(train_path)
print("Exists:", train_path.exists())

print("\nValidation images:")
print(val_path)
print("Exists:", val_path.exists())

print("\nTest images:")
print(test_path)
print("Exists:", test_path.exists())

assert train_path.exists()
assert val_path.exists()
assert test_path.exists()

print("\n✓ All image directories exist.")

YOLO DATASET STRUCTURE
Dataset root:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset

Train images:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\train
Exists: True

Validation images:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\val
Exists: True

Test images:
c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\images\test
Exists: True

✓ All image directories exist.


In [26]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

def count_images(folder):
    return len([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

train_count = count_images(train_path)
val_count = count_images(val_path)
test_count = count_images(test_path)

total_count = train_count + val_count + test_count

print("=" * 70)
print("PREPARED DATASET IMAGE COUNTS")
print("=" * 70)

print(f"Train      : {train_count}")
print(f"Validation : {val_count}")
print(f"Test       : {test_count}")
print(f"Total      : {total_count}")

print("=" * 70)

assert train_count == 4990
assert val_count == 1444
assert test_count == 776

print("✓ Dataset image counts match the prepared dataset.")

PREPARED DATASET IMAGE COUNTS
Train      : 4990
Validation : 1444
Test       : 776
Total      : 7210
✓ Dataset image counts match the prepared dataset.


## 3.6 Verify YOLO Label Files

Each image in the prepared dataset should have a corresponding YOLO annotation file.

YOLO annotation files use the `.txt` format.

Each annotation contains:

```text
class_id center_x center_y width height

In [27]:



# Cell 16 — Code


labels_root = dataset_root / "labels"

train_labels = labels_root / "train"
val_labels = labels_root / "val"
test_labels = labels_root / "test"

print("=" * 70)
print("YOLO LABEL DIRECTORIES")
print("=" * 70)

for name, folder in [
    ("Train", train_labels),
    ("Validation", val_labels),
    ("Test", test_labels)
]:
    print(f"\n{name}:")
    print("Path:", folder)
    print("Exists:", folder.exists())

    if folder.exists():
        label_count = len(list(folder.glob("*.txt")))
        print("Label files:", label_count)

assert train_labels.exists()
assert val_labels.exists()
assert test_labels.exists()

print("\n✓ All YOLO label directories exist.")

YOLO LABEL DIRECTORIES

Train:
Path: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\labels\train
Exists: True
Label files: 4990

Validation:
Path: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\labels\val
Exists: True
Label files: 1444

Test:
Path: c:\Users\USER\Desktop\computer vision\electronic-waste-detection-and-classification-for-automated-recycling-system-\data\yolo_dataset\labels\test
Exists: True
Label files: 776

✓ All YOLO label directories exist.


## 3.7 Load the YOLO26n Pretrained Model

YOLO26n will be initialized using pretrained weights.

Transfer learning is used rather than training the network from random initialization.

The pretrained model provides general visual features, which will subsequently be fine-tuned for the 37 electronic waste object classes.

The actual training process is performed in Phase 4.

In [28]:
from ultralytics import YOLO

MODEL_NAME = "yolo26n.pt"

print("Loading model:", MODEL_NAME)

model = YOLO(MODEL_NAME)

print("\n✓ YOLO26n model loaded successfully.")

Loading model: yolo26n.pt

✓ YOLO26n model loaded successfully.


In [29]:
print("=" * 70)
print("YOLO26n MODEL INFORMATION")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Task:", model.task)

print("\nModel architecture:")
print(model.model)

print("=" * 70)

YOLO26n MODEL INFORMATION
Model: yolo26n.pt
Task: detect

Model architecture:
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64,

In [30]:
print("Model task:")
print(model.task)

print("\nPretrained model class count:")
print(len(model.names))

print("\nPretrained model classes:")
print(model.names)

Model task:
detect

Pretrained model class count:
80

Pretrained model classes:
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'l

## 3.8 Verify Model–Dataset Compatibility

Before training, the following conditions are checked:

1. YOLO26n loads successfully.
2. The model is an object detection model.
3. The dataset configuration exists.
4. The dataset contains 37 classes.
5. Training images exist.
6. Validation images exist.
7. Test images exist.
8. Training labels exist.
9. Validation labels exist.
10. Test labels exist.

If all checks pass, the project is ready for Phase 4 — YOLO26n model training.

In [31]:
print("=" * 70)
print("YOLO26n PROJECT READINESS CHECK")
print("=" * 70)

checks = {
    "YOLO26n model loaded": model is not None,
    "Model task is detection": model.task == "detect",
    "Dataset YAML exists": DATA_YAML.exists(),
    "Number of classes = 37": nc == 37,
    "37 class names available": len(names) == 37,
    "Train directory exists": train_path.exists(),
    "Validation directory exists": val_path.exists(),
    "Test directory exists": test_path.exists(),
    "Train labels exist": train_labels.exists(),
    "Validation labels exist": val_labels.exists(),
    "Test labels exist": test_labels.exists(),
}

for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{status:6} | {check}")

print("=" * 70)

if all(checks.values()):
    print("✓ YOLO26n MODEL SETUP COMPLETED SUCCESSFULLY")
    print("✓ DATASET IS READY FOR TRAINING")
else:
    print("✗ One or more checks failed.")
    print("Review the failed checks before training.")

YOLO26n PROJECT READINESS CHECK
PASS   | YOLO26n model loaded
PASS   | Model task is detection
PASS   | Dataset YAML exists
PASS   | Number of classes = 37
PASS   | 37 class names available
PASS   | Train directory exists
PASS   | Validation directory exists
PASS   | Test directory exists
PASS   | Train labels exist
PASS   | Validation labels exist
PASS   | Test labels exist
✓ YOLO26n MODEL SETUP COMPLETED SUCCESSFULLY
✓ DATASET IS READY FOR TRAINING


# Phase 3 Conclusion

YOLO26n has been selected as the object detection model for the electronic waste detection and classification project.

The pretrained YOLO26n model was successfully loaded and the prepared dataset was verified.

### Confirmed

- YOLO26n model loaded
- Object detection task confirmed
- 37 electronic waste classes
- 4,991 training images
- 1,444 validation images
- 776 test images
- YOLO training labels available
- YOLO validation labels available
- YOLO test labels available
- `data.yaml` available
- Dataset structure verified

### Model Development Approach

The pretrained YOLO26n model will be fine-tuned on the prepared electronic waste dataset.

### Next Phase

## Phase 4 — YOLO26n Model Training

The next phase will train YOLO26n using the prepared electronic waste dataset.

Training results will be recorded for later evaluation, including:

- Training loss
- Validation loss
- Precision
- Recall
- mAP@50
- mAP@50:95
- Per-class performance
- Confusion matrix
- Training curves
- Best model weights